# Clase 9: Agrupamiento. Laboratorio

Objetivo: programar K-means desde cero, usarlo en datos reales y compararlo con DBSCAN y el agrupamiento jerárquico.

Este laboratorio tiene tres archivos:

| Archivo | Qué es |
|---|---|
| `agrupamiento.py` | Las funciones del laboratorio. Es el único archivo que se edita. |
| `utilidades.py` | Funciones de apoyo: los datos, el ciclo de K-means, las verificaciones y las gráficas. No hace falta modificarlo. |
| `lab.ipynb` | Este notebook: importa las funciones de `agrupamiento.py`, las verifica y las usa. |

Protocolo: editar una función en `agrupamiento.py`, guardar el archivo y volver a ejecutar la celda de verificación aquí.

> Si el notebook no ve los cambios en `agrupamiento.py`: guardar el archivo y volver a ejecutar la celda. Si aun así no los ve, reiniciar el kernel (*Kernel → Restart*) y ejecutar las celdas desde el principio.

## Preparación

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt

import agrupamiento
from utilidades import *

# Parte 1: K-means a mano

K-means repite dos pasos:

- **Asignar.** Cada ejemplo va al centroide más cercano: $c^{(i)} := \arg\min_j \|x^{(i)} - \mu_j\|^2$.
- **Mover.** Cada centroide va al promedio de los ejemplos de su grupo.

En el código, `X` tiene un ejemplo por fila, `centroids` un centroide por fila y `c` el grupo de cada ejemplo. Los grupos se numeran desde 0.

## Ejercicio 1a

Completar `assign_clusters` en `agrupamiento.py`.

<details><summary>Pista</summary>

```python
for j in range(k):
    distances[:, j] = ((X - centroids[j]) ** 2).sum(axis=1)
return np.argmin(distances, axis=1)
```
`X - centroids[j]` resta el centroide $j$ a cada fila de `X`.
</details>

In [ ]:
check_assign_clusters(agrupamiento.assign_clusters)

## Ejercicio 1b

Completar `update_centroids`: el centroide $j$ es el promedio de las filas de `X` cuyo grupo es $j$.

<details><summary>Pista</summary>

```python
for j in range(k):
    centroids[j] = X[c == j].mean(axis=0)
return centroids
```
</details>

In [ ]:
check_update_centroids(agrupamiento.update_centroids)

## Ejercicio 1c

Completar `distortion`: $J(c, \mu) = \sum_i \|x^{(i)} - \mu_{c^{(i)}}\|^2$.

<details><summary>Pista</summary>

`centroids[c]` repite el centroide de cada ejemplo, así que `X - centroids[c]` tiene la diferencia de cada ejemplo con su centroide:
```python
return float(((X - centroids[c]) ** 2).sum())
```
</details>

In [ ]:
check_distortion(agrupamiento.distortion)

## K-means completo

La función `kmeans` de `utilidades.py` junta las tres funciones: escoge $k$ ejemplos al azar como centroides iniciales y repite asignar y mover hasta que nada cambia. 

Estos datos tienen 300 puntos en el plano, sin etiquetas:

In [ ]:
X = make_blob_data()
plot_clusters(X, title="Los datos")
plt.show()

In [ ]:
result = kmeans(X, k=4, seed=6, verbose=True)
plot_kmeans_steps(X, result)
plot_J(result)

In [ ]:
if result is not None:
    J = result["J"]
    verify(all(b <= a + 1e-9 for a, b in zip(J, J[1:])), "J nunca sube, como dice la sección 1.3",
           "J subió en alguna iteración: revisar las tres funciones")

    from sklearn.cluster import KMeans
    sk = KMeans(n_clusters=4, n_init=10, random_state=0).fit(X)
    print(f"J de este laboratorio: {J[-1]:.1f}")
    print(f"J de scikit-learn:     {sk.inertia_:.1f}")

## Mínimos locales

La celda de abajo corre K-means con ocho inicializaciones distintas (una por semilla) y muestra la $J$ final de cada una.

Preguntas.

1. ¿Todas las semillas llegan al mismo resultado? ¿Cuáles se quedan en un mínimo local?
2. ¿Cómo se ve un mínimo local en la gráfica?
3. ¿Qué hace `n_init=10` en scikit-learn, y por qué evita este problema?

In [ ]:
final_J = plot_seeds(X, k=4, seeds=range(8))

# Parte 2: Elegir $k$ y comprimir una foto

## Los datos de vinos

178 vinos de una región de Italia, con 13 features químicas (alcohol, acidez, color, etc.). Los vinos son de tres variedades, pero la variedad **no** se usa para agrupar: solo se usa al final para ver si los grupos de K-means tienen sentido.

In [ ]:
X_wine, y_wine, feature_names = load_wine_data()
print(X_wine.shape)
print(f"{'feature':30} {'media':>9} {'desviación':>11}")
for name, m, s in zip(feature_names, X_wine.mean(axis=0), X_wine.std(axis=0)):
    print(f"{name:30} {m:9.2f} {s:11.2f}")

Pregunta. Mirar la columna de la desviación estándar. ¿Qué feature va a dominar las distancias si no se estandariza?

## Sin estandarizar

In [ ]:
elbow_silhouette(X_wine, title="Vinos, sin estandarizar")

## Ejercicio 2

Completar `standardize` en `agrupamiento.py`: restar a cada columna su media y dividirla entre su desviación estándar (sección 1.6 de las notas).

<details><summary>Pista</summary>

```python
return (X - X.mean(axis=0)) / X.std(axis=0)
```
</details>

In [ ]:
check_standardize(agrupamiento.standardize)

In [ ]:
Z_wine = run(agrupamiento.standardize, X_wine)
if Z_wine is not None:
    elbow_silhouette(Z_wine, title="Vinos, estandarizando")

Preguntas.

1. ¿Qué $k$ sugiere la silueta sin estandarizar? ¿Y estandarizando?
2. Sin estandarizar, la silueta es más alta. ¿Quiere decir que esos grupos son mejores? (Pista: ¿qué feature decide las distancias?)

Ahora sí se usan las variedades, solo para comparar: cada fila es una variedad y cada columna un grupo de K-means con $k = 3$. Si los grupos coinciden con las variedades, cada fila tiene casi todos sus vinos en una sola columna.

In [ ]:
from sklearn.cluster import KMeans

for title, data in [("sin estandarizar", X_wine), ("estandarizando", Z_wine)]:
    if data is None:
        continue
    c = KMeans(n_clusters=3, n_init=10, random_state=0).fit_predict(data)
    print(f"\n{title}")
    print("variedad   grupo 0  grupo 1  grupo 2")
    for v in range(3):
        print(f"{v:8}   " + "  ".join(f"{np.sum((y_wine == v) & (c == j)):7}" for j in range(3)))

## Comprimir una foto

Cada píxel es un punto en $\mathbb{R}^3$ (rojo, verde, azul). La función `compress` entrena **el K-means de este laboratorio** con 5000 píxeles y después le da a cada píxel el color de su centroide más cercano (sección 1.7 de las notas).

In [ ]:
photo = load_photo()
print(photo.shape)

images = {"original": (photo, None)}
for k in [2, 4, 8, 16]:
    images[f"k = {k}"] = compress(photo, k)
show_images(images)

Preguntas.

1. ¿Desde qué $k$ la foto se ve casi igual a la original?
2. La foto original guarda 24 bits por píxel (8 por color). Con $k = 16$ colores, ¿cuántos bits hacen falta por píxel? ¿Cuánto más pequeña queda la foto (sin contar la paleta)?

# Parte 3: Los tres algoritmos sobre los mismos datos (~20 min)

## Ejercicio 3a

Completar `compare_algorithms` en `agrupamiento.py`: agrupar `X` con K-means, DBSCAN y el jerárquico con enlace single de scikit-learn (secciones 1.6, 2.2 y 3.2 de las notas).

<details><summary>Pista</summary>

```python
return {
    "kmeans": KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(X),
    "dbscan": DBSCAN(eps=eps, min_samples=min_samples).fit_predict(X),
    "single": AgglomerativeClustering(n_clusters=k, linkage="single").fit_predict(X),
}
```
</details>

In [ ]:
check_compare_algorithms(agrupamiento.compare_algorithms)

## Ejercicio 3b: escoger `eps`

Los tres conjuntos de datos, con las features estandarizadas. Para cada uno, la curva de la distancia al 5.º vecino (sección 2.3 de las notas). Escoger un `eps` para cada conjunto, donde la curva empieza a subir rápido, y escribirlo en el diccionario `eps` de la celda siguiente.

In [ ]:
shapes = load_shapes()

fig, axs = plt.subplots(2, 3, figsize=(12, 6), gridspec_kw={"height_ratios": [1.2, 1]})
for (name, (X_shape, k)), top, bottom in zip(shapes.items(), axs[0], axs[1]):
    plot_clusters(X_shape, title=f"{name} ({k} grupos)", ax=top, size=7)
    k_distance_plot(X_shape, min_samples=5, ax=bottom)
plt.tight_layout()
plt.show()

In [ ]:
# Falta completar: un eps para cada conjunto, leído de las curvas de arriba
eps = {"bolas": 0.1, "lunas": 0.1, "círculos": 0.1}

results = run_comparison(shapes, eps)
plot_comparison(shapes, results)

Preguntas.

1. ¿Qué método acierta en cada conjunto de datos? ¿Coincide con la figura de la Parte 4 de las notas?
2. En las bolas, DBSCAN y el enlace single juntan dos grupos. ¿Por qué?
3. Probar en las lunas un `eps` mucho más pequeño y uno mucho más grande. ¿Qué pasa en cada caso?
4. Si los datos tienen valores atípicos, ¿qué método conviene? ¿Por qué?

## Reto

Cambiar `"single"` por `"ward"` en `compare_algorithms` y volver a ejecutar las dos celdas de arriba. ¿A cuál de los otros dos métodos se parece más el enlace de Ward? (Al terminar, volver a dejar `"single"`.)